In [1]:
from pyspark.sql import SparkSession, Window, functions as F

spark = SparkSession.builder.appName("03_gold").getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")

BUCKET = "taxis-chicago-grandes-datos-506301"
SILVER = f"gs://{BUCKET}/silver"
GOLD = f"gs://{BUCKET}/gold"

26/10/04 20:11:25 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
silver = spark.read.parquet(f"{SILVER}/taxi_trips")

# --- Dimensión de zonas: las 77 community areas de Chicago ---
ZONE_NAMES = ["Rogers Park", "West Ridge", "Uptown", "Lincoln Square", "North Center", "Lake View",
    "Lincoln Park", "Near North Side", "Edison Park", "Norwood Park", "Jefferson Park", "Forest Glen",
    "North Park", "Albany Park", "Portage Park", "Irving Park", "Dunning", "Montclare",
    "Belmont Cragin", "Hermosa", "Avondale", "Logan Square", "Humboldt Park", "West Town", "Austin",
    "West Garfield Park", "East Garfield Park", "Near West Side", "North Lawndale", "South Lawndale",
    "Lower West Side", "Loop", "Near South Side", "Armour Square", "Douglas", "Oakland", "Fuller Park",
    "Grand Boulevard", "Kenwood", "Washington Park", "Hyde Park", "Woodlawn", "South Shore", "Chatham",
    "Avalon Park", "South Chicago", "Burnside", "Calumet Heights", "Roseland", "Pullman",
    "South Deering", "East Side", "West Pullman", "Riverdale", "Hegewisch", "Garfield Ridge",
    "Archer Heights", "Brighton Park", "McKinley Park", "Bridgeport", "New City", "West Elsdon",
    "Gage Park", "Clearing", "West Lawn", "Chicago Lawn", "West Englewood", "Englewood",
    "Greater Grand Crossing", "Ashburn", "Auburn Gresham", "Beverly", "Washington Heights",
    "Mount Greenwood", "Morgan Park", "O'Hare", "Edgewater"]
zones = (spark.createDataFrame([(i + 1, n) for i, n in enumerate(ZONE_NAMES)], ["zone", "zone_name"])
         .withColumn("zone", F.col("zone").cast("int")))

centroids = (silver.groupBy(F.col("pickup_community_area").alias("zone"))
             .agg(F.avg("pickup_latitude").alias("lat"), F.avg("pickup_longitude").alias("lon")))
dim_zones = zones.join(centroids, "zone", "left")
dim_zones.coalesce(1).write.mode("overwrite").parquet(f"{GOLD}/dim_zones")

In [3]:
# --- 1) Agregación de negocio: viajes por zona, fecha y hora ---
agg = (silver.groupBy(F.col("pickup_community_area").alias("zone"),
                      F.col("trip_date").alias("date"), F.col("trip_hour").alias("hour"))
       .agg(F.count("*").alias("trips"),
            F.round(F.sum("trip_total"), 2).alias("revenue"),
            F.countDistinct("taxi_id").alias("active_taxis"),
            F.round(F.avg("trip_seconds") / 60, 1).alias("avg_minutes")))

In [4]:
# --- 2) Malla completa: una fila por zona-fecha-hora aunque haya 0 viajes ---
span = silver.agg(F.min("trip_date").alias("first_day"), F.max("trip_date").alias("last_day")).first()
calendar = (spark.sql(f"SELECT explode(sequence(to_date('{span['first_day']}'), to_date('{span['last_day']}'), "
                      "interval 1 day)) AS `date`")
            .crossJoin(spark.range(24).select(F.col("id").cast("int").alias("hour"))))
grid = calendar.crossJoin(F.broadcast(zones))
base = (grid.join(agg, ["zone", "date", "hour"], "left")
        .na.fill(0, ["trips", "revenue", "active_taxis"]))

In [5]:
# --- 3) Variables de calendario ---
dow, dom, mon = F.dayofweek("date"), F.dayofmonth("date"), F.month("date")
is_holiday = (((mon == 1) & (dom == 1)) | ((mon == 7) & (dom == 4)) | ((mon == 12) & (dom == 25))
              | ((mon == 5) & (dow == 2) & (dom >= 25))               # Memorial Day
              | ((mon == 9) & (dow == 2) & (dom <= 7))                # Labor Day
              | ((mon == 11) & (dow == 5) & dom.between(22, 28)))     # Thanksgiving
base = (base.withColumn("year", F.year("date"))
        .withColumn("month", mon)
        .withColumn("day_of_week", dow)                               # 1 = domingo ... 7 = sábado
        .withColumn("is_weekend", dow.isin(1, 7).cast("int"))
        .withColumn("is_holiday", is_holiday.cast("int")))

In [6]:
# --- 4) Cruce con clima (ya limpio en silver) ---
weather = spark.read.parquet(f"{SILVER}/weather")
w_month = Window.partitionBy("year", "month")
base = (base.join(F.broadcast(weather), "date", "left")
        .withColumn("temp_c", F.coalesce("temp_c", F.avg("temp_c").over(w_month)))
        .na.fill({"prcp_mm": 0.0, "snow": 0}))

In [7]:
# --- 5) Perfil histórico: promedio de la misma zona y hora, 1 a 4 semanas atrás ---
w = Window.partitionBy("zone", "hour").orderBy("date")
lags = [F.lag("trips", k).over(w) for k in (7, 14, 21, 28)]
gold = (base.withColumn("profile_4w", (lags[0] + lags[1] + lags[2] + lags[3]) / 4)
        .filter(F.col("profile_4w").isNotNull()))

(gold.repartition("year").write.mode("overwrite").partitionBy("year")
 .parquet(f"{GOLD}/zone_hour_demand"))

In [8]:
# --- 6) Segunda tabla de negocio: KPIs diarios por empresa y tipo de pago ---
kpis = (silver.groupBy(F.col("trip_date").alias("date"), "company", "payment_type")
        .agg(F.count("*").alias("trips"),
             F.round(F.sum("trip_total"), 2).alias("revenue"),
             F.round(F.avg("trip_seconds") / 60, 1).alias("avg_minutes"),
             F.countDistinct("taxi_id").alias("active_taxis")))
kpis.coalesce(8).write.mode("overwrite").parquet(f"{GOLD}/daily_kpis")

In [9]:
# --- 7) Agregados pequeños para el tablero de la app ---
EXPORT = f"{GOLD}/app_export"
demand = spark.read.parquet(f"{GOLD}/zone_hour_demand")

In [10]:
def export(df, name):
    df.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{EXPORT}/{name}")

export(demand.groupBy("date")
       .agg(F.sum("trips").alias("trips"), F.round(F.sum("revenue"), 2).alias("revenue"),
            F.first("temp_c").alias("temp_c"), F.first("prcp_mm").alias("prcp_mm"),
            F.first("snow").alias("snow")),
       "dashboard_daily")
export(demand.groupBy("year", "zone", "zone_name", "day_of_week", "hour")
       .agg(F.sum("trips").alias("trips"), F.round(F.sum("revenue"), 2).alias("revenue")),
       "dashboard_zone_hour")
export(spark.read.parquet(f"{GOLD}/daily_kpis")
       .groupBy(F.year("date").alias("year"), "company")
       .agg(F.sum("trips").alias("trips"), F.round(F.sum("revenue"), 2).alias("revenue")),
       "dashboard_companies")

In [11]:
demand = spark.read.parquet(f"{GOLD}/zone_hour_demand")
print(f"Filas: {demand.count():,}")

demand.agg(
    F.min("date").alias("first_day"),
    F.max("date").alias("last_day"),
    F.countDistinct("zone").alias("zones"),
    F.sum("trips").alias("trips"),
    F.sum(F.col("temp_c").isNull().cast("int")).alias("temp_nulls"),
    F.sum(F.col("profile_4w").isNull().cast("int")).alias("profile_nulls"),
).show()

demand.groupBy("year").agg(F.sum("trips").alias("trips")).orderBy("year").show()

zones = spark.read.parquet(f"{GOLD}/dim_zones")
print(f"Zonas: {zones.count()} | sin coordenadas: {zones.filter('lat IS NULL').count()}")
print(f"Filas en daily_kpis: {spark.read.parquet(f'{GOLD}/daily_kpis').count():,}")

Filas: 7,371,672


+----------+----------+-----+---------+----------+-------------+
| first_day|  last_day|zones|    trips|temp_nulls|profile_nulls|
+----------+----------+-----+---------+----------+-------------+
|2013-01-29|2023-12-31|   77|141175103|         0|            0|
+----------+----------+-----+---------+----------+-------------+



+----+--------+
|year|   trips|
+----+--------+
|2013|20533812|
|2014|30521190|
|2015|19597100|
|2016|15530763|
|2017|10313038|
|2018|11490599|
|2019|14790671|
|2020| 3462949|
|2021| 3445170|
|2022| 5554772|
|2023| 5935039|
+----+--------+

Zonas: 77 | sin coordenadas: 0


Filas en daily_kpis: 379,022


In [12]:
spark.stop()